In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


INVENTORY_PATH = PROJECT_ROOT / "data" / "inventory" / "inventory.parquet"

SAMPLE_DIR = PROJECT_ROOT / "data" / "samples"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 120
RANDOM_STATE = 42

In [2]:
inventory = pd.read_parquet(INVENTORY_PATH)

print(f"Documentos no inventário: {len(inventory)}")
inventory.head()

Documentos no inventário: 3429


,document_id,sha256,path,relative_path,filename,stem,extension,size_bytes,size_mb,created_at,...,page_count,encrypted,pdf_version,title,author,subject,creator,producer,status,error
0,415c1ac896f59486,415c1ac896f594865c2f1f427d7b4f504e5a402b32f063...,D:\baseia_v3\corpus\00 - Preço de Liquidação d...,00 - Preço de Liquidação das Diferenças_2023.3...,00 - Preço de Liquidação das Diferenças_2023.3...,00 - Preço de Liquidação das Diferenças_2023.3...,.pdf,1890470,1.8029,2026-07-27T23:59:49.566359+00:00,...,25.0,False,1.7,Microsoft Word - 00 - Preço de Liquidação das ...,rcarneiro,NaN,NaN,Microsoft: Print To PDF,ok,NaN
1,ff4ca23d0f16d57f,ff4ca23d0f16d57f82f5b2446f2ae3dafc72166d260a07...,D:\baseia_v3\corpus\010+n7+BJD.pdf,010+n7+BJD.pdf,010+n7+BJD.pdf,010+n7+BJD,.pdf,575632,0.5490,2026-07-28T00:00:16.726151+00:00,...,19.0,False,1.3,ISSN: 2525-8761,Priscila Viégas,NaN,Microsoft® Word 2019,Microsoft® Word 2019,ok,NaN
2,ce2c31a603353dbd,ce2c31a603353dbdac0bde49e43b3149ada42e15aba926...,D:\baseia_v3\corpus\0605582v2.pdf,0605582v2.pdf,0605582v2.pdf,0605582v2,.pdf,335288,0.3198,2026-07-28T00:00:16.798797+00:00,...,24.0,False,1.4,Bayesian analysis for reversible Markov chains,"Persi Diaconis, Silke W. W. Rolles","The Annals of Statistics, 2006, Vol.34, No.3, ...",LaTeX with hyperref package,dvips + GPL Ghostscript GIT PRERELEASE 9.22,ok,NaN
3,39f7076074f617aa,39f7076074f617aa8d5c6d6c281f19ba49104c58ef2904...,D:\baseia_v3\corpus\1-s2.0-S0169207014001083-m...,1-s2.0-S0169207014001083-main.pdf,1-s2.0-S0169207014001083-main.pdf,1-s2.0-S0169207014001083-main,.pdf,3041424,2.9005,2026-07-28T00:00:16.799798+00:00,...,52.0,False,1.7,Electricity price forecasting: A review of the...,Rafał Weron,"International Journal of Forecasting, 30 (2014...",Elsevier,pdfTeX-1.40.3,ok,NaN
4,974bbe22ff347665,974bbe22ff347665410c629724e75ee83af3108a9fdef4...,D:\baseia_v3\corpus\1-s2.0-S2666546824000880-m...,1-s2.0-S2666546824000880-main.pdf,1-s2.0-S2666546824000880-main.pdf,1-s2.0-S2666546824000880-main,.pdf,4962478,4.7326,2026-07-28T00:00:16.728151+00:00,...,14.0,False,1.7,Probabilistic simulation of electricity price ...,Viktor Walter,"Energy and AI, 18 (2024) 100422. doi:10.1016/j...",Elsevier,Acrobat Distiller 8.1.0 (Windows),ok,NaN


In [3]:
eligible = inventory[
    inventory["status"].eq("ok")
    & inventory["sha256"].notna()
    & inventory["path"].notna()
].copy()

eligible = (
    eligible.sort_values("relative_path")
    .drop_duplicates(
        subset="sha256",
        keep="first",
    )
    .reset_index(drop=True)
)

eligible["page_count"] = pd.to_numeric(
    eligible["page_count"],
    errors="coerce",
)

eligible["size_mb"] = pd.to_numeric(
    eligible["size_mb"],
    errors="coerce",
)

print(f"Documentos elegíveis e únicos: {len(eligible)}")

Documentos elegíveis e únicos: 3380


In [4]:
eligible["page_bucket"] = pd.cut(
    eligible["page_count"],
    bins=[
        -np.inf,
        2,
        5,
        10,
        25,
        50,
        100,
        250,
        np.inf,
    ],
    labels=[
        "001_002",
        "003_005",
        "006_010",
        "011_025",
        "026_050",
        "051_100",
        "101_250",
        "251_plus",
    ],
)

eligible["size_bucket"] = pd.cut(
    eligible["size_mb"],
    bins=[
        -np.inf,
        0.25,
        1,
        5,
        20,
        100,
        np.inf,
    ],
    labels=[
        "000_025mb",
        "025_001mb",
        "001_005mb",
        "005_020mb",
        "020_100mb",
        "100mb_plus",
    ],
)

eligible["encrypted_bucket"] = (
    eligible["encrypted"]
    .fillna(False)
    .map(
        {
            False: "not_encrypted",
            True: "encrypted",
        }
    )
)

eligible["stratum"] = (
    eligible["page_bucket"].astype(str)
    + "|"
    + eligible["size_bucket"].astype(str)
    + "|"
    + eligible["encrypted_bucket"]
)

In [5]:
strata = (
    eligible.groupby("stratum", observed=True)
    .size()
    .rename("population")
    .sort_values(ascending=False)
    .reset_index()
)

strata

,stratum,population
0,006_010|025_001mb|not_encrypted,1207
1,006_010|001_005mb|not_encrypted,1010
2,011_025|025_001mb|not_encrypted,357
3,011_025|001_005mb|not_encrypted,287
4,006_010|000_025mb|not_encrypted,115
5,251_plus|005_020mb|not_encrypted,87
6,026_050|025_001mb|not_encrypted,41
7,003_005|025_001mb|not_encrypted,31
8,011_025|000_025mb|not_encrypted,28
9,251_plus|001_005mb|not_encrypted,28


In [6]:
def stratified_sample(
    dataframe: pd.DataFrame,
    *,
    sample_size: int,
    random_state: int,
) -> pd.DataFrame:
    if sample_size >= len(dataframe):
        return dataframe.copy()

    groups = list(
        dataframe.groupby(
            "stratum",
            observed=True,
            sort=True,
        )
    )

    minimum_per_stratum = 1

    selected_parts = []

    for _, group in groups:
        selected_parts.append(
            group.sample(
                n=min(minimum_per_stratum, len(group)),
                random_state=random_state,
            )
        )

    selected = pd.concat(
        selected_parts,
        ignore_index=False,
    ).drop_duplicates("sha256")

    remaining_slots = sample_size - len(selected)

    if remaining_slots <= 0:
        return selected.sample(
            n=sample_size,
            random_state=random_state,
        ).reset_index(drop=True)

    remaining = dataframe[~dataframe["sha256"].isin(selected["sha256"])].copy()

    population_weights = (
        remaining.groupby("stratum", observed=True).size().div(len(remaining))
    )

    additional_parts = []

    for stratum, group in remaining.groupby(
        "stratum",
        observed=True,
        sort=True,
    ):
        target = round(population_weights.loc[stratum] * remaining_slots)

        target = min(
            max(target, 0),
            len(group),
        )

        if target:
            additional_parts.append(
                group.sample(
                    n=target,
                    random_state=random_state,
                )
            )

    if additional_parts:
        selected = pd.concat(
            [selected, *additional_parts],
            ignore_index=False,
        ).drop_duplicates("sha256")

    missing = sample_size - len(selected)

    if missing > 0:
        remaining = dataframe[~dataframe["sha256"].isin(selected["sha256"])]

        selected = pd.concat(
            [
                selected,
                remaining.sample(
                    n=min(missing, len(remaining)),
                    random_state=random_state,
                ),
            ],
            ignore_index=False,
        )

    if len(selected) > sample_size:
        selected = selected.sample(
            n=sample_size,
            random_state=random_state,
        )

    return selected.sort_values(
        [
            "page_bucket",
            "size_bucket",
            "relative_path",
        ]
    ).reset_index(drop=True)

In [7]:
sample = stratified_sample(
    eligible,
    sample_size=SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Documentos na amostra: {len(sample)}")
sample.head()

Documentos na amostra: 120


,document_id,sha256,path,relative_path,filename,stem,extension,size_bytes,size_mb,created_at,...,author,subject,creator,producer,status,error,page_bucket,size_bucket,encrypted_bucket,stratum
0,dd9a0ad9cbad7321,dd9a0ad9cbad73216c444a26be38a5438c6a8428342a9c...,D:\baseia_v3\corpus\metadata_williamson_1999_b...,metadata_williamson_1999_bureaucracies.pdf,metadata_williamson_1999_bureaucracies.pdf,metadata_williamson_1999_bureaucracies,.pdf,2605,0.0025,2026-07-28T00:00:16.782798+00:00,...,O. Williamson,(unspecified),(unspecified),ReportLab PDF Library - (opensource),ok,NaN,001_002,000_025mb,not_encrypted,001_002|000_025mb|not_encrypted
1,4a9063205fe01dcd,4a9063205fe01dcde5109f7d14c338f4005a429c27ff53...,D:\baseia_v3\corpus\Risk-Sharing Contracts and...,Risk-Sharing Contracts and risk management of ...,Risk-Sharing Contracts and risk management of ...,Risk-Sharing Contracts and risk management of ...,.pdf,950003,0.9060,2026-07-28T00:22:00.993326+00:00,...,paula.candeias,NaN,NaN,Microsoft: Print To PDF,ok,NaN,001_002,025_001mb,not_encrypted,001_002|025_001mb|not_encrypted
2,64fde0a612b00eda,64fde0a612b00eda1caf1ee45f4711d9d69cb182481f26...,D:\baseia_v3\corpus\New approach to financial ...,New approach to financial time series forecast...,New approach to financial time series forecast...,New approach to financial time series forecast...,.pdf,241117,0.2299,2026-07-28T00:20:43.888672+00:00,...,brchang,NaN,PScript5.dll Version 5.2.2,Acrobat Distiller 7.0.5 (Windows),ok,NaN,003_005,000_025mb,not_encrypted,003_005|000_025mb|not_encrypted
3,c01077e63ee98d45,c01077e63ee98d459d6b834d323f29320b83174cc21032...,D:\baseia_v3\corpus\uerj_rede_sirius_procedime...,uerj_rede_sirius_procedimentos_bdtd.pdf,uerj_rede_sirius_procedimentos_bdtd.pdf,uerj_rede_sirius_procedimentos_bdtd,.pdf,44391,0.0423,2026-07-27T23:57:48.208418+00:00,...,sirius,NaN,PScript5.dll Version 5.2,Acrobat Distiller 7.0.5 (Windows),ok,NaN,003_005,000_025mb,encrypted,003_005|000_025mb|encrypted
4,169ba37b912aa8dd,169ba37b912aa8ddf44315dce0edf8e7670219d53d5154...,D:\baseia_v3\corpus\GTL_0081.pdf,GTL_0081.pdf,GTL_0081.pdf,GTL_0081,.pdf,970548,0.9256,2026-07-28T00:02:50.986700+00:00,...,Gabriel Magalhaes,NaN,NaN,Microsoft: Print To PDF,ok,NaN,003_005,025_001mb,not_encrypted,003_005|025_001mb|not_encrypted


In [9]:
edge_parts = [
    eligible.nlargest(10, "size_mb"),
    eligible.nlargest(10, "page_count"),
    eligible.nsmallest(10, "size_mb"),
    eligible[
        eligible["encrypted"].fillna(False)
    ].head(10),
]

edge_cases = (
    pd.concat(edge_parts, ignore_index=True)
    .drop_duplicates("sha256")
)

edge_cases = edge_cases[
    ~edge_cases["sha256"].isin(sample["sha256"])
].reset_index(drop=True)

print(f"Casos extremos adicionais: {len(edge_cases)}")

Casos extremos adicionais: 26


In [10]:
manifest_columns = [
    "document_id",
    "sha256",
    "path",
    "relative_path",
    "filename",
    "size_mb",
    "page_count",
    "encrypted",
    "page_bucket",
    "size_bucket",
    "stratum",
]

SAMPLE_MANIFEST = SAMPLE_DIR / "benchmark_sample.csv"
EDGE_CASES_MANIFEST = SAMPLE_DIR / "benchmark_edge_cases.csv"

sample[manifest_columns].to_csv(
    SAMPLE_MANIFEST,
    index=False,
    encoding="utf-8-sig",
)

edge_cases[manifest_columns].to_csv(
    EDGE_CASES_MANIFEST,
    index=False,
    encoding="utf-8-sig",
)

print(SAMPLE_MANIFEST.resolve())
print(EDGE_CASES_MANIFEST.resolve())

D:\baseia_v3\data\samples\benchmark_sample.csv
D:\baseia_v3\data\samples\benchmark_edge_cases.csv


In [11]:
display(
    sample["page_bucket"].value_counts(
        sort=False,
        dropna=False,
    ),
    sample["size_bucket"].value_counts(
        sort=False,
        dropna=False,
    ),
    sample["stratum"].value_counts(),
)

page_bucket
001_002      2
003_005      5
006_010     67
011_025     22
026_050      7
051_100      5
101_250      4
251_plus     8
Name: count, dtype: int64

size_bucket
000_025mb     11
025_001mb     53
001_005mb     45
005_020mb      9
020_100mb      2
100mb_plus     0
Name: count, dtype: int64

stratum
006_010|025_001mb|not_encrypted     32
006_010|001_005mb|not_encrypted     27
011_025|025_001mb|not_encrypted     10
011_025|001_005mb|not_encrypted      8
006_010|000_025mb|not_encrypted      4
251_plus|005_020mb|not_encrypted     3
003_005|025_001mb|not_encrypted      2
006_010|005_020mb|not_encrypted      2
011_025|000_025mb|not_encrypted      2
026_050|025_001mb|not_encrypted      2
026_050|001_005mb|not_encrypted      2
101_250|001_005mb|not_encrypted      2
251_plus|001_005mb|not_encrypted     2
251_plus|020_100mb|not_encrypted     2
001_002|000_025mb|not_encrypted      1
001_002|025_001mb|not_encrypted      1
003_005|000_025mb|not_encrypted      1
003_005|000_025mb|encrypted          1
003_005|001_005mb|not_encrypted      1
006_010|000_025mb|encrypted          1
006_010|025_001mb|encrypted          1
011_025|001_005mb|encrypted          1
011_025|005_020mb|not_encrypted      1
026_050|000_025mb|not_encrypted      1
026_050|025_001mb|encrypted          1
026_050|005_020mb